# 02 — Calidad y limpieza

Cada transformación sigue el esquema **evidencia → decisión → impacto** y se registra en
`logs/pipeline_log.csv` (columnas: Paso, Descripción, Filas, Nulos, Retención (%)).
El dataset original permanece intacto en `data/raw/`; el resultado se guarda en `data/processed/`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_json("../data/raw/streaming_users_dirty.json")
FILAS_INICIALES = len(df)

log = []
def registrar(paso, descripcion):
    log.append({
        "Paso": paso,
        "Descripción": descripcion,
        "Filas": len(df),
        "Nulos": int(df.isna().sum().sum()),
        "Retención (%)": round(len(df) / FILAS_INICIALES * 100, 2),
    })

registrar(0, "Estado inicial (data/raw)")
log[-1]

{'Paso': 0,
 'Descripción': 'Estado inicial (data/raw)',
 'Filas': 8160,
 'Nulos': 753,
 'Retención (%)': 100.0}

## Paso 1 — Duplicados exactos

**Evidencia (notebook 01):** 126 filas idénticas en todas sus columnas.
**Decisión:** eliminarlas conservando la primera aparición; una fila idéntica no aporta información
y sesga los conteos.

In [2]:
antes = len(df)
df = df.drop_duplicates()
registrar(1, "Eliminación de duplicados exactos")
print(f"Filas eliminadas: {antes - len(df)} | Filas restantes: {len(df)}")

Filas eliminadas: 126 | Filas restantes: 8034


## Paso 2 — Duplicados por `user_id`

**Evidencia:** aún quedan `user_id` repetidos (copias casi idénticas con alguna celda alterada).
Cada usuario debe aparecer una sola vez.
**Decisión:** conservar la primera aparición de cada `user_id`.

In [3]:
antes = len(df)
print("user_id repetidos antes:", df["user_id"].duplicated().sum())
df = df.drop_duplicates(subset="user_id", keep="first")
registrar(2, "Eliminación de duplicados por user_id (se conserva la primera aparición)")
print(f"Filas eliminadas: {antes - len(df)} | Filas restantes: {len(df)}")

user_id repetidos antes: 34
Filas eliminadas: 34 | Filas restantes: 8000


## Paso 3 — Normalización de `subscription_plan`

**Evidencia:** 15 variantes de escritura para 3 planes reales (mayúsculas, inglés, "Std", "Premiun",
espacios finales).
**Decisión:** mapear todas las variantes a 3 categorías canónicas: Básico, Estándar, Premium.

In [4]:
MAPA_PLAN = {
    "basico": "Básico", "básico": "Básico", "basic": "Básico",
    "estandar": "Estándar", "estándar": "Estándar", "std": "Estándar", "standard": "Estándar",
    "premium": "Premium", "premiun": "Premium",
}
antes = df["subscription_plan"].nunique()
df["subscription_plan"] = df["subscription_plan"].str.strip().str.lower().map(MAPA_PLAN)
assert df["subscription_plan"].notna().all(), "quedaron planes sin mapear"
registrar(3, "Normalización de subscription_plan (15 variantes -> 3 categorías)")
print(f"Categorías: {antes} -> {df['subscription_plan'].nunique()}")
df["subscription_plan"].value_counts()

Categorías: 15 -> 3


subscription_plan
Básico      3600
Estándar    2817
Premium     1583
Name: count, dtype: int64

## Paso 4 — Normalización de `country`

**Evidencia:** 26 variantes para 7 países (nombres en dos idiomas, códigos ISO, minúsculas, espacios).
**Decisión:** mapear a los 7 nombres canónicos en español.

In [5]:
MAPA_PAIS = {
    "argentina": "Argentina", "arg": "Argentina",
    "brasil": "Brasil", "brazil": "Brasil", "bra": "Brasil",
    "chile": "Chile", "chl": "Chile",
    "colombia": "Colombia", "col": "Colombia",
    "mexico": "México", "méxico": "México", "mex": "México",
    "peru": "Perú", "perú": "Perú", "per": "Perú",
    "uruguay": "Uruguay", "ury": "Uruguay",
}
antes = df["country"].nunique()
df["country"] = df["country"].str.strip().str.lower().map(MAPA_PAIS)
assert df["country"].notna().all(), "quedaron países sin mapear"
registrar(4, "Normalización de country (26 variantes -> 7 países)")
print(f"Categorías: {antes} -> {df['country'].nunique()}")
df["country"].value_counts()

Categorías: 26 -> 7


country
Chile        1164
México       1156
Brasil       1156
Uruguay      1143
Colombia     1142
Perú         1134
Argentina    1105
Name: count, dtype: int64

## Paso 5 — Normalización de `favorite_genre` y tratamiento de sus faltantes

**Evidencia:** 28 variantes para 7 géneros, más 240 valores nulos (~3 %).
**Decisión:** mapear a 7 géneros canónicos. Los nulos se recategorizan como **"Desconocido"** en lugar
de eliminarse o imputarse: la preferencia no se puede inferir y borrar las filas perdería el resto de la
información del usuario.

In [6]:
MAPA_GENERO = {
    "acción": "Acción", "accion": "Acción", "action": "Acción",
    "comedia": "Comedia", "comedy": "Comedia",
    "crime": "Crimen", "crimen": "Crimen",
    "doc": "Documental", "documental": "Documental", "documentary": "Documental",
    "drama": "Drama",
    "romance": "Romance",
    "thriller": "Thriller", "thriler": "Thriller",
}
nulos_antes = df["favorite_genre"].isna().sum()
df["favorite_genre"] = df["favorite_genre"].str.strip().str.lower().map(MAPA_GENERO)
df["favorite_genre"] = df["favorite_genre"].fillna("Desconocido")
registrar(5, "Normalización de favorite_genre (28 variantes -> 7 géneros) y nulos -> 'Desconocido'")
print(f"Nulos recategorizados como 'Desconocido': {nulos_antes}")
df["favorite_genre"].value_counts()

Nulos recategorizados como 'Desconocido': 240


favorite_genre
Comedia        1137
Drama          1115
Romance        1110
Documental     1107
Thriller       1104
Acción         1103
Crimen         1084
Desconocido     240
Name: count, dtype: int64

## Paso 6 — Edades imposibles

**Evidencia:** valores -5, 0, 4, 130 y 150. Un servicio de streaming exige una edad plausible; se define
el rango válido **[13, 100]** (13 = edad mínima habitual de registro).
**Decisión:** marcar como faltante lo que queda fuera del rango e imputar con la **mediana** (robusta a
la asimetría y afecta ~1,5 % de las filas; eliminarlas perdería el resto de la información del usuario).

In [7]:
fuera = ((df["age"] < 13) | (df["age"] > 100)).sum()
df.loc[(df["age"] < 13) | (df["age"] > 100), "age"] = np.nan
mediana_edad = df["age"].median()
df["age"] = df["age"].fillna(mediana_edad)
registrar(6, f"age fuera de [13,100] ({fuera} casos) -> imputación con mediana ({mediana_edad:.0f})")
print(f"Valores corregidos: {fuera} | Mediana usada: {mediana_edad:.0f}")
print(f"Rango final de age: [{df['age'].min():.0f}, {df['age'].max():.0f}]")

Valores corregidos: 120 | Mediana usada: 33
Rango final de age: [13, 80]


## Paso 7 — Tiempo de visualización inválido y faltante

**Evidencia:** 49 valores negativos (imposibles), 31 valores 99999 (centinela: se separa en 4 órdenes de
magnitud del percentil 75 ≈ 1046 y la cola real llega a ~3300) y 193 nulos originales.
**Decisión:** convertir negativos y centinelas en faltantes e imputar todos los faltantes con la
**mediana por plan de suscripción**: el consumo depende fuertemente del plan, por lo que la mediana del
propio plan es un mejor estimador que la mediana global.

In [8]:
invalidos = ((df["monthly_watch_time_mins"] < 0) | (df["monthly_watch_time_mins"] >= 10000)).sum()
nulos_antes = df["monthly_watch_time_mins"].isna().sum()
df.loc[(df["monthly_watch_time_mins"] < 0) | (df["monthly_watch_time_mins"] >= 10000),
       "monthly_watch_time_mins"] = np.nan
medianas_plan = df.groupby("subscription_plan")["monthly_watch_time_mins"].median()
df["monthly_watch_time_mins"] = df.groupby("subscription_plan")["monthly_watch_time_mins"] \
    .transform(lambda s: s.fillna(s.median()))
registrar(7, f"watch_time: {invalidos} inválidos (negativos/99999) + {nulos_antes} nulos -> mediana por plan")
print(f"Inválidos convertidos: {invalidos} | Nulos originales: {nulos_antes}")
print("Medianas por plan usadas para imputar:")
print(medianas_plan.round(1))

Inválidos convertidos: 80 | Nulos originales: 193
Medianas por plan usadas para imputar:
subscription_plan
Básico       552.7
Estándar     840.0
Premium     1127.0
Name: monthly_watch_time_mins, dtype: float64


## Paso 8 — Tickets de soporte inválidos

**Evidencia:** 29 valores -1 (centinela de faltante) y 67 valores 99/150. La distribución real va de 0 a 5
(mediana 1): 99 y 150 no son actividad plausible de un cliente en el período, y por aparecer solo esos dos
valores exactos repetidos se interpretan como centinelas/carga errónea y no como usuarios extremos reales.
**Decisión:** convertir ambos grupos en faltantes e imputar con la mediana.

In [9]:
invalidos = ((df["customer_support_tickets"] < 0) | (df["customer_support_tickets"] > 30)).sum()
df.loc[(df["customer_support_tickets"] < 0) | (df["customer_support_tickets"] > 30),
       "customer_support_tickets"] = np.nan
mediana_tickets = df["customer_support_tickets"].median()
df["customer_support_tickets"] = df["customer_support_tickets"].fillna(mediana_tickets)
registrar(8, f"tickets inválidos (-1, 99, 150: {invalidos} casos) -> imputación con mediana ({mediana_tickets:.0f})")
print(f"Valores corregidos: {invalidos} | Distribución final: 0 a {df['customer_support_tickets'].max():.0f}")

Valores corregidos: 96 | Distribución final: 0 a 5


## Paso 9 — Fechas de último inicio de sesión

**Evidencia:** dos formatos mezclados (AAAA-MM-DD y DD-MM-AAAA), valores inválidos (`0000-00-00`,
`31-02-2022`, `not_available`, `2026-15-03`) y fechas futuras (`2029-01-01`). Fecha de referencia del
análisis: **2026-07-04**.
**Decisión:** parsear ambos formatos; lo inválido y lo futuro pasa a faltante (`NaT`). **No se eliminan
filas**: la fecha es una variable más y el resto del registro sigue siendo válido.

In [10]:
FECHA_REF = pd.Timestamp("2026-07-04")
s = df["last_login_date"].astype("string")
fechas = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
recuperadas = pd.to_datetime(s, format="%d-%m-%Y", errors="coerce")
print("Fechas recuperadas del formato DD-MM-AAAA:", int((fechas.isna() & recuperadas.notna()).sum()))
fechas = fechas.fillna(recuperadas)
futuras = (fechas > FECHA_REF).sum()
fechas[fechas > FECHA_REF] = pd.NaT
df["last_login_date"] = fechas
registrar(9, f"Fechas: parseo de 2 formatos; inválidas y futuras ({futuras}) -> NaT (sin eliminar filas)")
print(f"Fechas futuras anuladas: {futuras}")
print(f"Fechas faltantes finales: {df['last_login_date'].isna().sum()} "
      f"({df['last_login_date'].isna().mean()*100:.1f} %)")

Fechas recuperadas del formato DD-MM-AAAA: 174
Fechas futuras anuladas: 15
Fechas faltantes finales: 610 (7.6 %)


## Paso 10 — Variable derivada: `days_since_last_login`

**Decisión:** derivar los días transcurridos desde el último inicio de sesión hasta la fecha de
referencia. Aporta una medida numérica de **recencia** utilizable en el EDA y en PCA
(una fecha cruda no es apta para esas técnicas). Queda faltante donde la fecha es faltante.

In [11]:
df["days_since_last_login"] = (FECHA_REF - df["last_login_date"]).dt.days
registrar(10, "Derivación de days_since_last_login (recencia respecto de 2026-07-04)")
df[["last_login_date", "days_since_last_login"]].describe()

,last_login_date,days_since_last_login
count,7390,7390.000000
mean,2022-01-29 17:44:30.527740416,1616.260758
min,2018-01-01 00:00:00,185.000000
25%,2020-01-22 00:00:00,876.000000
50%,2022-02-12 12:00:00,1602.500000
75%,2024-02-09 00:00:00,2355.000000
max,2025-12-31 00:00:00,3106.000000
std,NaN,845.004817


## Guardado del dataset procesado y del log

In [12]:
df.to_csv("../data/processed/streaming_users_clean.csv", index=False)
log_df = pd.DataFrame(log)
log_df.to_csv("../logs/pipeline_log.csv", index=False)
log_df

,Paso,Descripción,Filas,Nulos,Retención (%)
0,0,Estado inicial (data/raw),8160,753,100.00
1,1,Eliminación de duplicados exactos,8034,753,98.46
2,2,Eliminación de duplicados por user_id (se cons...,8000,753,98.04
3,3,Normalización de subscription_plan (15 variant...,8000,753,98.04
4,4,Normalización de country (26 variantes -> 7 pa...,8000,753,98.04
5,5,Normalización de favorite_genre (28 variantes ...,8000,513,98.04
6,6,"age fuera de [13,100] (120 casos) -> imputació...",8000,513,98.04
7,7,watch_time: 80 inválidos (negativos/99999) + 1...,8000,320,98.04
8,8,"tickets inválidos (-1, 99, 150: 96 casos) -> i...",8000,320,98.04
9,9,Fechas: parseo de 2 formatos; inválidas y futu...,8000,610,98.04


## Resumen del impacto

- Se eliminaron **160 filas** (solo duplicados): retención del **98 %**.
- Ninguna fila se eliminó por faltantes o valores inválidos: se prefirió imputar (mediana / mediana por
  plan) o recategorizar ("Desconocido", `NaT`) para conservar la información restante de cada usuario.
- Las tres categóricas quedaron con sus categorías reales (3 planes, 7 países, 7+1 géneros).
- Los únicos faltantes remanentes están en `last_login_date` / `days_since_last_login` (~7,6 %),
  documentados y tratados explícitamente en las etapas siguientes.